# 05 — Unified Generator Benchmark

This validation-only notebook answers RQ1. It keeps the 1,361-image requirement on each synthetic candidate pool while using **all available positive validation images** as the real reference pool. Every metric repetition is balanced and records its sampled IDs. The test split is forbidden.

## 1. Protocol configuration

In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.generator_benchmark import (
    FEATURE_SPACES, REPRESENTATIONS, discover_candidates, duplicate_diagnostics,
    evaluation_subset_size, extract_features, get_or_extract_embeddings,
    load_protocol, load_registry, metadata_positive_paths, nearest_neighbours,
    repeated_distribution_metrics, render_similarity_panel, similarity_summaries,
    synthetic_nearest_neighbours, technical_audit,
)
protocol = load_protocol(ROOT)
registry = load_registry(ROOT)
RUN_FEATURE_EXTRACTION = False
protocol

## 2. Candidate discovery

In [ ]:
candidate_audits = discover_candidates(ROOT, protocol, registry)
candidate_audits

## 3. Candidate eligibility

In [ ]:
eligible = [row for row in candidate_audits if row['eligible_for_benchmark_execution']]
[(row['generator_id'], row['candidate_role'], row['eligible_for_downstream_selection'], row['blockers']) for row in candidate_audits]

## 4. RAW/FILTERED image counts

In [ ]:
counts = [{
    'generator_id': row['generator_id'],
    **{name: row['representations'][name]['count'] for name in REPRESENTATIONS}
} for row in candidate_audits]
counts

## 5. Real reference set

In [ ]:
validation_paths, validation_ids = metadata_positive_paths(ROOT, 'data/processed/metadata/val.csv')
real_reference_count = len(validation_paths)
{'real_reference_count': real_reference_count, 'policy': 'all available validation positives'}

## 6. Technical validity

In [ ]:
technical_results = {}
if RUN_FEATURE_EXTRACTION:
    for row in eligible:
        entry = next(item for item in registry['generators'] if item['id'] == row['generator_id'])
        for representation in REPRESENTATIONS:
            paths = sorted((ROOT / entry['samples'][f'{representation}_positive']).glob('*'))
            technical_results[(row['generator_id'], representation)] = technical_audit(paths)
technical_results

## 7. Feature extraction

In [ ]:
embedding_cache_root = ROOT / protocol['embedding_cache']['root']
embedding_metadata = {}
# Extraction happens once per generator × RAW/FILTERED × extractor and is reused below.
if RUN_FEATURE_EXTRACTION:
    for row in eligible:
        entry = next(item for item in registry['generators'] if item['id'] == row['generator_id'])
        for representation in REPRESENTATIONS:
            image_paths = sorted((ROOT / entry['samples'][f'{representation}_positive']).glob('*'))
            image_ids = [path.name for path in image_paths]
            for extractor_name in FEATURE_SPACES:
                cache = embedding_cache_root / row['generator_id'] / representation / f'{extractor_name}.npy'
                _, metadata = get_or_extract_embeddings(
                    cache, image_paths, image_ids, extractor=extractor_name,
                    preprocessing='registered frozen extractor preprocessing', code_version='informational git commit',
                    source_manifest=entry['provenance_manifest'],
                    extract_fn=lambda paths, name: extract_features(paths, name, allow_model_download=False),
                )
                embedding_metadata[(row['generator_id'], representation, extractor_name)] = metadata
embedding_metadata

## 8. Distribution metrics

In [ ]:
metric_summaries, repetition_records = {}, []
# KID=200, PRDC=100 and FID=1 by default; every subset is balanced and replace=False.
def compute_distribution_summary(real_features, synthetic_features):
    records, summary = repeated_distribution_metrics(real_features, synthetic_features, protocol)
    return records, summary
if RUN_FEATURE_EXTRACTION:
    print('Load each cached real/synthetic feature pair and call compute_distribution_summary.')
{'resampling': protocol['resampling'], 'summaries': metric_summaries}

## 9. Diversity analysis

In [ ]:
diversity_results = {}  # LPIPS/MS-SSIM or registered pairwise diversity, computed from deterministic pairs
diversity_results

## 10. Duplicate analysis

In [ ]:
duplicate_results = {}
if RUN_FEATURE_EXTRACTION:
    for row in eligible:
        entry = next(item for item in registry['generators'] if item['id'] == row['generator_id'])
        duplicate_results[row['generator_id']] = duplicate_diagnostics(sorted((ROOT / entry['samples']['filtered_positive']).glob('*')))
duplicate_results

## 11. Train memorization analysis

In [ ]:
train_memorization_rows = []  # synthetic → nearest real training positive; this alone may trigger the memorization gate
train_memorization_rows[:5]

## 12. Validation similarity analysis

In [ ]:
validation_similarity_rows = []  # synthetic → nearest real validation positive; descriptive, never a memorization gate
validation_similarity_rows[:5]

## 13. Bootstrap/repeated subsampling

In [ ]:
example_sizes = {
    row['generator_id']: evaluation_subset_size(row['representations']['filtered']['count'], real_reference_count, protocol['synthetic_pool_target'])
    for row in eligible
}
{'evaluation_subset_size_by_generator': example_sizes,
 'sampling': 'balanced without replacement',
 'recorded_fields': ['repetition', 'seed', 'real_indices', 'synthetic_indices']}

## 14. Results tables

In [ ]:
results_table = []  # populated only from computed benchmark result files; no placeholder numbers
results_table

## 15. Pareto analysis

In [ ]:
pareto_columns = ['RAD-DINO KID (min)', 'coverage (max)', 'precision (max)', 'train memorization (min)', 'efficiency']
pareto_columns

## 16. Visual panels

In [ ]:
panel_policy = ['closest / median / farthest synthetic vs train', 'closest / median / farthest synthetic vs validation', 'closest / median / farthest synthetic vs synthetic']
# render_similarity_panel is called separately for train, validation and synthetic-neighbour rows.
{'policy': panel_policy, 'renderer': render_similarity_panel.__name__}

## 17. Family-specific conclusions

Conclusions are written only after the result table exists. The 50-step Stable Diffusion row is a sampling ablation beside its 100-step counterpart, not an independent eligible winner. The first LDM remains a descriptive historical baseline until lineage is demonstrated. FID is secondary and explicitly unstable at the small real-reference count.